# Mobilising needed libraries & 1st Checking of the datasets

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import time
import warnings
import multiprocessing 
from datetime import date

# Use all but one CPU core for parallel processing
num_cores = multiprocessing.cpu_count() - 1
print("Using", num_cores, "cores for parallel processing.")

warnings.filterwarnings("ignore")

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical utilities
import scipy.stats as stats
from scipy.stats import chi2, chi2_contingency, f_oneway   # ← your missingness tests

# Statsmodels (ANOVA, OLS)
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Preprocessing & feature engineering
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelling algorithms
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# Model evaluation & selection
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Optional (dimension reduction)
from sklearn.decomposition import PCA

In [ ]:
# Importing the training dataset
train = pd.read_csv("ML_WP_data/train.csv")
# Checking the table
train.head()
# Calculate total number of NaN values in the DataFrame
total_train_nans = train.isna().sum().sum()

# Display the total count of NaN values
print("Total NaN values in the DataFrame:", total_train_nans)

In [ ]:
train.info()

# Exploratory Data Analysis

## Overview of Missing Values and Descriptive Statistics in the Dataset - CC

In [ ]:
# Creating a summary DataFrame for variables with missing values
missing_summary = (
    pd.DataFrame({
        'Data_Type': train.dtypes,  # Get the data type of each column
        'Missing_Values': train.isnull().sum(),  # Calculate the total number of missing values in each column
        'Percent_Missing': (train.isnull().sum() / len(train) * 100).round(2)  # Calculate the percentage of missing values and round it to two decimal places
    })
    .query("Missing_Values > 0")  # Filter for columns that have missing values
    .sort_values(by='Missing_Values', ascending=False)  # Sort by the number of missing values, in descending order
)

# Generating a statistical summary of the dataset (similar to what .describe() provides)
summary_stats = train.describe(include='all').transpose()  # Transpose to have features as rows for easier viewing

# Merging the missing summary with descriptive statistics for better insights on the data
merged_summary = missing_summary.merge(
    summary_stats[['mean', 'std', 'min', '25%', '50%', '75%', 'max']],  # Select statistical measures to include
    left_index=True,  # Use the index of the missing summary
    right_index=True,  # Match it with the index of the summary statistics
    how='left'  # Perform a left join to keep all rows from missing summary
)

# Printing the merged summary, converting it to a string for better readability
print(merged_summary.to_string())

# Printing the total number of variables that have missing values
print(f"\nTotal variables with missing values: {len(missing_summary)}")

# Identifying all target columns, specifically the temperature variables to be predicted
target_columns = train.filter(like='target').columns.tolist()  # Filter columns that include 'target' in their names


## Key observations of the Missing Value Summary and Dispersion Analysis



* **Precipitation (`rre150h0_*`)** – *rre150h0: Precipitation; hourly total*  
  * Most hourly values are zero (25%, 50%, 75% = 0).  
  * Maximum values up to 41.7 mm/hour (e.g., Lugano) are realistic for Swiss storms.  
  * Very small means and large standard deviations indicate strong right skew.  
  * These “outliers” are genuine rainfall events, not errors.

* **Global radiation (`gre000h0_*`)** – *gre000h0: Global radiation; hourly mean*  
  * Many zeros at night; daytime values reach up to around 1000 W/m².  
  * High standard deviation (~250 W/m²) reflects day–night cycles.  
  * Strongly skewed, possibly bimodal distribution (night vs. day).

* **Atmospheric pressure (`prestah0_*`)** – *prestah0: Atmospheric pressure at barometric altitude (QFE); hourly mean*  
  * Narrow ranges (~7 hPa std) consistent with expected altitudes.  
  * No missing or implausible values detected.  
  * Excellent sensor consistency across stations.

* **Temperature (`tre200h0_*`)** – *tre200h0: Temperature at 2 m above ground; hourly mean*  
  * Mean temperature varies by station (≈ 4 °C in Davos/Andermatt, ≈ 13 °C in Lugano).  
  * Ranges from −23 °C to +38 °C, plausible for Swiss seasonal extremes.  
  * Differences in means due to elevation, not data quality issues.

* **Humidity (`ure200h0_*`)** – *ure200h0: Relative air humidity 2 m above ground; hourly mean*  
  * Mostly within 0–100 %; rare minima (≈ 2–3 %) possible in very dry air.  
  * Some near-100 % readings are expected under saturated conditions.  
  * Physically bounded variable; clipping can be applied if needed.

* **Wind (`fkl010h*_*`)** – *fkl010h1: Gust peak (one second); hourly maximum in m/s*  
  * Substantial variability; some locations (La Dôle, Andermatt) show strong gusts.  
  * Distributions are right-skewed with high maxima but physically plausible.  
  * Reflects natural geographic differences rather than anomalies.

🎯 **Target columns in the dataset:**
- target_tre200h0_plus12h  
- target_tre200h0_plus24h  
- target_tre200h0_plus48h  

### Preprocessing recommendations

| Variable group                                       | Issues identified                    | Suggested action                                                |
| ---------------------------------------------------- | ------------------------------------ | --------------------------------------------------------------- |
| **Pressure**                                         | None                                 | Keep as is                                                      |
| **Temperature**                                      | Different baselines between stations | Apply Z-score normalisation per station                         |
| **Humidity**                                         | Physically bounded (0–100 %)         | Clip values to [0, 100]                                         |
| **Precipitation, Radiation, Wind**                   | Heavy right skew, many zeros         | Optionally apply log(1 + x) transformation before linear models |
| **Targets (`target_tre200h0_+12h`, `+24h`, `+48h`)** | No missing values                    | Keep as dependent variables                                     |

| Code  | Station    | Region    | Terrain type        |
|-------|-------------|-----------|---------------------|
| BAS   | Basel       | North     | Lowland             |
| GVE   | Genève      | West      | Lowland             |
| INT   | Interlaken  | Central   | Valley              |
| SIO   | Sion        | South     | Alpine valley       |
| STG   | St-Gallen   | Northeast | Foothills           |
| DAV   | Davos       | East      | High Alpine         |
| ZER   | Zermatt     | South     | High Alpine         |
| ANT   | Andermatt   | Central   | Mountain pass       |
| DOL   | La Dôle     | West      | Jura mountains      |
| LUG   | Lugano      | South     | Pre-Alpine / Ticino |

In [ ]:
# Select only numeric columns
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()

# If you want to exclude targets, do it here:
targets = ["target_tre200h0_plus12h", "target_tre200h0_plus24h", "target_tre200h0_plus48h"]
numeric_cols = [c for c in numeric_cols if c not in targets]

# Function to detect outliers using the IQR rule
def detect_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    return outliers

# Apply to all numeric columns
outlier_summary = {}
for col in numeric_cols:
    n_outliers = len(detect_outliers(train, col))
    outlier_summary[col] = n_outliers

# Convert to DataFrame for easier analysis
outlier_df = pd.DataFrame(list(outlier_summary.items()), columns=["Variable", "Outlier_Count"])
outlier_df["Outlier_%"] = (outlier_df["Outlier_Count"] / len(train)) * 100
outlier_df.sort_values(by="Outlier_%", ascending=False, inplace=True)

display(outlier_df.head(20))  # show top 15 variables with most outliers

Como vemos en la tabla, hay variables con gran porcentaje de observaciones consideradas extremas. Sin embargo, al comprobar la localización geográfica de esa observación, podemos concluir que más que valores extremos propios de un error de medición, estos valores se deben a la gran variabilidad geográfica y por ende meteorológica de Suiza.
En otras palabras, el IQR no toma en consideración el lugar en el que las mediciones fueron tomadas. Para referencia, hemos incluido una tabla con las localizaciones de las estaciones meteorológicas de esta base de datos.
En conclusión, no recomendamos eliminar los valores extremos, ya que, por ejemplo, una lluvia intensa en Davos, podría predecir una caída de temperaturas en Berna en las próximas 24h.

Winsorisation is not recommended either since the outliers are likely a real phenomenon and not measurement errors and winsorising would distort the weather distribution.

pair.plot de seaborn

## Visualizations of Target Variables: Distribution, Boxplot, and Q-Q Analysis

In [ ]:
# List of target columns to visualize
targets = ["target_tre200h0_plus12h", "target_tre200h0_plus24h", "target_tre200h0_plus48h"]

# Create subplots: each row corresponds to a target variable, with 3 columns for different plots
fig, axes = plt.subplots(len(targets), 3, figsize=(15, 12))

# Iterate through each target column to create visualizations
for i, target_col in enumerate(targets):
    # Histogram for the target variable's distribution
    axes[i, 0].hist(train[target_col], bins=50, edgecolor='black', alpha=0.7)
    axes[i, 0].set_title(f"{target_col} – Distribution")  # Title for the histogram
    axes[i, 0].set_xlabel("Temperature (°C)")  # Label for the x-axis
    axes[i, 0].set_ylabel("Frequency")  # Label for the y-axis

    # Boxplot for the target variable to visualize median and quartiles
    axes[i, 1].boxplot(train[target_col].dropna(), vert=True)  # Drop NaN values for the boxplot
    axes[i, 1].set_title(f"{target_col} – Boxplot")  # Title for the boxplot
    axes[i, 1].set_ylabel("Temperature (°C)")  # Label for the y-axis

    # Q-Q plot to assess normality of the target variable
    stats.probplot(train[target_col].dropna(), dist="norm", plot=axes[i, 2])  # Normal distribution comparison
    axes[i, 2].set_title(f"{target_col} – Q–Q Plot vs Normal")  # Title for the Q-Q plot

# Adjust layout to prevent overlap of subplots
plt.tight_layout()

# Show the combined plots
plt.show()


## Skewness and Kurtosis Analysis of Target Variables

In [ ]:
# List of target columns to analyze for skewness and kurtosis
targets = ["target_tre200h0_plus12h", "target_tre200h0_plus24h", "target_tre200h0_plus48h"]

# Iterate through each target variable to compute skewness and kurtosis
for t in targets:
    clean_series = train[t].dropna()  # Remove missing values from the target series
    clean_series = clean_series[np.isfinite(clean_series)]  # Remove any infinite values (inf/-inf)

    # Calculate skewness to assess the asymmetry of the distribution
    skew = stats.skew(clean_series)
    
    # Calculate kurtosis to assess the "tailedness" of the distribution
    kurt = stats.kurtosis(clean_series)

    # Print the results in a formatted string
    print(f"{t}: Skewness = {skew:.3f}, Kurtosis = {kurt:.3f}")


Los resultados que vimos en el histograma, boxplot y QQplot de arriba son confirmados por el análisis de skwenness. Al estar las tres variables objetivos con una skewness baja (sobre los 0.2 puntos), esto nos indica que hay ligreamente más temperaturas sobre cero que inferiores, cosa dentro de lo normal en mediciones de temperatura.
Respecto a la kurtosis, vemos como las tres variables tienen una ligera kurtosis negativa, indicando que las distribuciones son algo más planas y con colas más ligeras que una distribución normal.

## Analysis of Missing Values by Hour with Chi-Square Test

In [ ]:
# Calculate the count of missing values for each row in the train DataFrame
train['missing_count'] = train.isnull().sum(axis=1)

# Group the data by hour and compute the total missing values for each hour
missing_by_hour = (
    train.groupby('hour')['missing_count']
    .sum()  # Sum total missing values for each hour
    .sort_values(ascending=False)  # Sort hours by total missing values in descending order
)

# Display the top 10 hours with the most missing values
print("Top hours with most missing values:\n")
print(missing_by_hour.head(10))


# Plot the distribution of missing values by hour of the day
plt.figure(figsize=(10, 4))
missing_by_hour.sort_index().plot(kind='bar', color='steelblue', edgecolor='black')  # Bar plot
plt.title("Total Missing Values by Hour of the Day")  # Title of the plot
plt.xlabel("Hour (0–23)")  # X-axis label
plt.ylabel("Number of Missing Values")  # Y-axis label
plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()  # Display the plot

# Count total missing values per hour again for later analysis
missing_by_hour = train.groupby('hour')['missing_count'].sum()

# Chi-square goodness-of-fit test to assess if missing values differ by hour
# Null hypothesis: all hours have an equal count of missing values
expected = [missing_by_hour.sum() / len(missing_by_hour)] * len(missing_by_hour)  # Expected counts
chi2_stat = ((missing_by_hour - expected)**2 / expected).sum()  # Calculate chi-square statistic

# Calculate p-value from chi-square statistic
p_value = 1 - chi2.cdf(chi2_stat, df=len(missing_by_hour) - 1)

# Display chi-square statistic and p-value
print(f"Chi-square statistic: {chi2_stat:.2f}")
print(f"p-value: {p_value:.4f}")

# Determine significance of the results
if p_value < 0.05:
    print("→ Missingness differs significantly by hour (reject H₀).")
else:
    print("→ No significant hourly difference (fail to reject H₀).")


Como vemos en el gráfico de barras de arriba, los valores ausentes parecen más o menos uniformemente distribuidos a través de las horas del día, con tan solo pequeñas fluctuaciones y un ligero pico a las 13h. En general, podríamos decir que no hay patrones horarios sistemáticos en la falta de observaciones. Esto es confirmado mediante el test de chi-cuadrado, con una p-value superior a 0.05, en otras palabras, no podemos rechazar la hipótesis nula de independencia entre hora y ausencia de observaciones. En conclusión, estos datos podrían ser descritos como Missing Completetly at Random (MCAR) respecto a la hora del día.

## Seasonal Analysis of Missing Values with Chi-Square and ANOVA Tests

In [ ]:
# --- Seasonal analysis of missing values (only if 'season' column exists) ---

# Check that the DataFrame actually contains a 'season' column
if 'season' in train.columns:
    # Define a logical order of the seasons for nicer plotting and reading
    season_order = ['Winter', 'Spring', 'Summer', 'Autumn']

    # Group the data by season and sum the total number of missing values per season
    # Then reindex the result so that the rows follow the desired season order
    missing_by_season = (
        train.groupby('season')['missing_count']
        .sum()  # Sum the total missing values for each season
        .reindex(season_order)  # Reindex to ensure the order of seasons
    )

    # Print the total missing values per season to the console
    print("\nMissing values by season:")
    print(missing_by_season)

    # Create a bar plot showing total missing values per season
    plt.figure(figsize=(6, 4))  # Set the figure size for the plot
    missing_by_season.plot(
        kind='bar',            # Create a bar chart
        color='darkorange',    # Set the bar colour
        edgecolor='black'      # Set the bar border colour
    )
    plt.title("Missing Values by Season")     # Add a title to the plot
    plt.xlabel("Season")                      # Label for the x-axis
    plt.ylabel("Total Missing Values")        # Label for the y-axis
    plt.tight_layout()                        # Adjust layout to avoid overlaps
    plt.show()                                # Display the plot

    # --- Chi-square test: Missing values by season ---

    # Recompute the grouped totals (no reindex needed, we just need counts)
    missing_by_season = train.groupby('season')['missing_count'].sum()

    # Under the null hypothesis, missingness is equally distributed across seasons.
    # Compute the expected count per season: total_missing / number_of_seasons.
    total_missing = missing_by_season.sum()              # Total number of missing values across all seasons
    n_seasons = len(missing_by_season)                   # Number of distinct seasons
    expected = [total_missing / n_seasons] * n_seasons   # Create a list of expected counts for each season

    # Compute the chi-square statistic manually:
    # sum( (observed - expected)^2 / expected ) for all seasons.
    chi2_stat = ((missing_by_season - expected) ** 2 / expected).sum()

    # Compute the p-value using the chi-square distribution CDF.
    # Degrees of freedom = number_of_seasons - 1.
    df = n_seasons - 1
    p_value = 1 - chi2.cdf(chi2_stat, df=df)  # Calculate the p-value

    # Print the results of the chi-square test
    print("=== Chi-square Test for Seasonal Missingness ===")
    print(f"Chi-square statistic: {chi2_stat:.2f}")  # Display chi-square statistic
    print(f"Degrees of freedom: {df}")                # Display degrees of freedom
    print(f"p-value: {p_value:.4f}")                  # Display p-value

    # Interpret the result at the 5% significance level
    if p_value < 0.05:
        print("→ Missingness differs significantly by season (reject H₀ of equal distribution).")
    else:
        print("→ No significant seasonal difference in missingness (fail to reject H₀).")

    # --- ANOVA: Does the mean missing_count differ by season? ---

    # Fit a linear model where 'missing_count' is explained by the categorical variable 'season'
    # C(season) tells statsmodels to treat 'season' as a categorical factor.
    model = ols('missing_count ~ C(season)', data=train).fit()

    # Perform an ANOVA (Type II sums of squares) on the fitted model
    anova_table = sm.stats.anova_lm(model, typ=2)

    # Print the ANOVA table to inspect whether season explains variance in missing_count
    print("\n=== ANOVA Test for Seasonal Missingness ===")
    print(anova_table)

else:
    # If 'season' is not present, inform the user and skip seasonal analysis
    print("Column 'season' not found in 'train': skipping seasonal missingness analysis.")


Como se observa en el gráfico anterior, los valores ausentes son más frecuentes en verano, con un total de 47, y en otoño, con 40. La ausencia de datos es independiente de la estación en la que se producen, ya que el test chi-cuadrado presenta un p-valor superior a 0.05; por tanto, no se rechaza la hipótesis nula $H_0$. En consecuencia, la ausencia de datos es aleatoria y despreciable, como muestran los números absolutos de valores faltantes. En otras palabras, estos valores ausentes representan ruido aleatorio.

A modo de conclusión, tanto los análisis por hora como por estación indican que las observaciones ausentes están distribuidas de manera aleatoria en el tiempo. Los test chi-cuadrado y ANOVA muestran que no existen diferencias estadísticamente significativas entre horas ni estaciones (p-valor superior a 0.05). Por ello, las observaciones ausentes son consistentes con un patrón de tipo Missing Completely at Random (MCAR). Esto justifica el uso de técnicas de imputación simple, ya que la ausencia de observaciones y su posterior imputación no introducirían sesgos en el entrenamiento de los modelos estadísticos.


# Data Preprocessing: Handling Missing Values and Creating Imputed Datasets

In [ ]:
# --- Step 1: Preserve original and define base datasets ---

# Create a copy of the original training dataset to avoid modifying it
base = train.copy()

# Identify numeric and categorical columns in the dataset
num_cols = base.select_dtypes(include='number').columns  # Numeric columns
cat_cols = base.select_dtypes(exclude='number').columns  # Categorical columns

# 1) Drop rows with any NA values
train_drop = base.dropna()  # Create a dataset without any missing values

# 2) Mean-imputed dataset
mean_imputer = SimpleImputer(strategy='mean')  # Initialize the imputer to fill missing values with the mean
train_mean_imp = base.copy()  # Create a copy of the base dataset
# Apply mean imputation to numeric columns
train_mean_imp[num_cols] = mean_imputer.fit_transform(train_mean_imp[num_cols])

# If there are any categorical columns, fill their missing values with the mode
if len(cat_cols) > 0:
    cat_modes = train_mean_imp[cat_cols].mode().iloc[0]  # Calculate modes for categorical columns
    train_mean_imp[cat_cols] = train_mean_imp[cat_cols].fillna(cat_modes)  # Fill NAs with modes

# 3) Median-imputed dataset
median_imputer = SimpleImputer(strategy='median')  # Initialize the imputer for median substitution
train_median_imp = base.copy()  # Create another copy of the base dataset
# Apply median imputation to numeric columns
train_median_imp[num_cols] = median_imputer.fit_transform(train_median_imp[num_cols])

# Fill missing values in categorical columns with the mode, if applicable
if len(cat_cols) > 0:
    cat_modes = train_median_imp[cat_cols].mode().iloc[0]  # Calculate modes for categorical columns
    train_median_imp[cat_cols] = train_median_imp[cat_cols].fillna(cat_modes)  # Fill NAs with modes

# Sanity checks to ensure no unexpected missing values remain
print(f"Rows before drop: {len(base)}, after drop: {len(train_drop)}")  # Count of rows before and after dropping
print("Remaining NAs after drop:", train_drop.isnull().sum().sum())  # Total remaining NAs after dropping
print("Remaining NAs after mean imputation:", train_mean_imp.isnull().sum().sum())  # Check remaining NAs after mean imputation
print("Remaining NAs after median imputation:", train_median_imp.isnull().sum().sum())  # Check remaining NAs after median imputation

# --- Step 2: Build datasets dictionary (NO scaling, NO PCA) ---
# Store the different datasets in a dictionary for easy access
datasets = {
    'drop_na': train_drop,  # Dataset with NAs dropped
    'mean_imputed': train_mean_imp,  # Dataset after mean imputation
    'median_imputed': train_median_imp  # Dataset after median imputation
}


## Correlation Matrix Visualization for Numeric Variables

In [ ]:
# Compute the correlation matrix for all numeric variables in the training dataset
corr = train.corr(numeric_only=True)  # Use numeric_only=True to include only numeric columns in the correlation calculation

# Set up the figure for displaying the heatmap
plt.figure(figsize=(12, 10))  # Specify the size of the figure

# Create a heatmap to visualize the correlation matrix
sns.heatmap(corr, cmap="coolwarm", center=0)  # Use "coolwarm" color palette to represent correlation values

# Add a title to the heatmap for clarity
plt.title("Correlation Matrix — All Numeric Variables")

# Display the heatmap
plt.show()


El el gráfico de correlaciones arriba, vemos como hay una gran correlación intravariable en cada estación meteorológica como en la variable gre000h0 (que mide la radiación) o en prestah0 (que mide la presión atmosférica). Esto no es anómalo, puesto que las estaciones meteorológicas son espacialmente coherentes, en otras palabras, las temperaturas y presiones atmosféricas se mueven de manera armoniosa. Del mismo modo, hay correlaciones cruzadas débiles entre variables. Por ejemplo, la variable tre200h0 (temperatura) está correlacionada positivamente con gre000h0 (radiación) y negativamente con ure200h0 (humedad). De igual modo, rre150h0(precipitaciones) está correlacionada negativamente con gre000h0 (radiación), lo que nos indica consistencia en términos meteorológicos (los valores de las variables están dentro de los rangos esperados para Suiza)

Sin embargo, la alta correlación entre variables implica multicolinealidad, es decir, la presencia de información redundante entre predictores que puede distorsionar el ajuste de ciertos modelos. Esta multicolinearidad no afecta a la capacidad predictiva de nuestros modelos, pero si podría inflar la varianza de los coeficientes y desestabilizar los modelos lineales o de regresión. Para neutralizar este impacto negativo, se podrían aplicar un Principal Analysis Components (PCA) con la cuál reduciríamos la dimensionalidad de los datos. También se podrían eliminar las variables altamente correlacionadas, evitando así redundancias.

# Visualizing Correlation with Target Variable in DataFrame

In [ ]:
def plot_top_correlations(df, target_col='target_tre200h0_plus24h', k=6,
                          exclude_cols=('hour', 'season',
                                        'target_tre200h0_plus12h',
                                        'target_tre200h0_plus24h',
                                        'target_tre200h0_plus48h')):
    # Select numeric predictors only, excluding specified columns (targets and helpers)
    num_cols = df.select_dtypes(include=np.number).columns  # Get all numeric columns
    pred_cols = [c for c in num_cols if c not in exclude_cols]  # Filter out excluded columns

    # Calculate absolute correlations with the target column and sort them
    corrs = df[pred_cols].corrwith(df[target_col]).abs().sort_values(ascending=False)
    top_feats = corrs.head(k).index.tolist()  # Select the top k features based on correlation

    # Create a grid layout for subplots (2 rows x 3 cols if k=6)
    rows = int(np.ceil(k / 3))  # Calculate the number of rows needed
    cols = 3 if k >= 3 else k  # Set the number of columns
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows))  # Create subplots
    axes = np.atleast_1d(axes).flatten()  # Flatten the axes array for easy iteration

    # Plot each of the top features against the target variable
    for ax, col in zip(axes, top_feats):
        # Clean NaN values for plotting
        x = df[col].values  # Predictor values
        y = df[target_col].values  # Target values
        m = ~np.isnan(x) & ~np.isnan(y)  # Boolean mask for non-NaN values
        x, y = x[m], y[m]  # Filter NaN values

        # Create a scatter plot
        ax.scatter(x, y, alpha=0.6, s=15)  # Scatter plot with transparency and size

        # Compute and plot the Ordinary Least Squares (OLS) regression line
        if x.size >= 2:  # Ensure there are enough points for fitting
            z = np.polyfit(x, y, 1)  # Fit a degree-1 polynomial (line)
            p = np.poly1d(z)  # Create a polynomial function from the coefficients
            xx = np.linspace(x.min(), x.max(), 200)  # Create a range for plotting the line
            ax.plot(xx, p(xx), "r--", alpha=0.8, lw=2)  # Plot the regression line

        # Calculate the Pearson correlation coefficient
        r = np.corrcoef(x, y)[0, 1] if x.size > 1 else np.nan  # Calculate correlation
        ax.set_title(f"{col} vs {target_col}\nCorrelation: {r:.3f}")  # Set title
        ax.set_xlabel(col)  # Set x-axis label
        ax.set_ylabel(target_col)  # Set y-axis label

    # Hide any empty axes (if k is not a multiple of 3)
    for ax in axes[len(top_feats):]:
        ax.axis('off')  # Turn off the axes

    plt.tight_layout()  # Adjust layout to prevent overlap
    plt.show()  # Display the plots

# ---- Use it on your dataframe (choose which version) ----
# df = train_mean_imp  # or train_median_imp, train_drop, or train
plot_top_correlations(train, target_col='target_tre200h0_plus24h', k=6)


# 0. PCA Analysis on Scaled Datasets: Variance Visualization and Summary

In [ ]:
# ============================================
# PCA variance plots + tables for each dataset
# (using the datasets built in Step 1)
# ============================================

# If not already defined:
target_cols = [
    "target_tre200h0_plus12h",
    "target_tre200h0_plus24h",
    "target_tre200h0_plus48h"
]

feature_prefixes = ['fkl010h', 'gre000h0', 'rre150h0', 'prestah0', 'ure200h0', 'tre200h0']

cols_to_scale = [
    c for c in base.columns
    if any(p in c for p in feature_prefixes) and c not in target_cols
]

for label in ['drop_na', 'mean_imputed', 'median_imputed']:
    print(f"\n--- PCA Variance Plot and Table for: {label} ---")

    df = datasets[label].copy()
    X = df[cols_to_scale].copy()

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA()
    X_pca = pca.fit_transform(X_scaled)

    explained_var = pca.explained_variance_ratio_
    cumulative_var = np.cumsum(explained_var)
    eigenvalues = pca.explained_variance_

    plt.figure(figsize=(5, 3))
    plt.plot(explained_var, marker='o', label='Explained variance')
    plt.plot(cumulative_var, marker='s', label='Cumulative variance')
    plt.title(f'Explained & Cumulative Variance — PCA ({label})')
    plt.xlabel('Number of Components')
    plt.ylabel('Proportion of Variance')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    start_idx = 34  # 35th component
    end_idx = 45    # up to 45th (exclusive)

    summary_df = pd.DataFrame({
        'Component': np.arange(1, len(eigenvalues) + 1),
        'EigenValue': eigenvalues,
        'Variance (%)': explained_var * 100,
        'Cumulative Variance (%)': cumulative_var * 100
    })

    print(summary_df.loc[start_idx:end_idx - 1].round(4))
    print("Number of features in PCA:", len(cols_to_scale))
    print("Any targets included?:", any(t in cols_to_scale for t in target_cols))


explain like in AD3M what eigen values and communalities are¡¡¡

AS the plots and tables suggest, we will keep 37 components

# 0.1 Applying PCA to Scaled Datasets with Sanity Check

In [ ]:
#  pca_n_components = 37

# for label in ['drop_na', 'mean_imputed', 'median_imputed']:
    #df_scaled = datasets[label].copy()

    # Apply PCA only on scaled columns
    #pca = PCA(n_components=pca_n_components)
    # X_pca = pca.fit_transform(df_scaled[cols_to_scale])

    # pca_cols = [f'pca_{i+1}' for i in range(pca_n_components)]
    # X_pca_df = pd.DataFrame(X_pca, columns=pca_cols, index=df_scaled.index)

    # Drop scaled columns, keep targets + others
    # df_pca = df_scaled.drop(columns=cols_to_scale)
    # df_pca = pd.concat([df_pca, X_pca_df], axis=1)

    # Ensure targets are included
    # for tgt in target_cols:
        # if tgt not in df_pca.columns and tgt in df_scaled.columns:
            # df_pca[tgt] = df_scaled[tgt]

    #datasets[f"{label}_pca"] = df_pca

# =====================================================
# 4️⃣  Sanity check
# =====================================================
# for name in ['mean_imputed', 'mean_imputed_pca']:
    # print(f"\n{name}:")
    # print(datasets[name].filter(like='target').columns.tolist())

# 2. Model Definitions and Hyperparameter Grids for PCA and Regression Techniques

In [ ]:
# === Models ===
models = {
    'Linear Regression': LinearRegression(),  # Simple linear regression model
    'Ridge': Ridge(),  # Ridge regression with L2 regularization
    'Lasso': Lasso(),  # Lasso regression with L1 regularization
    'Random Forest': RandomForestRegressor(random_state=42),  # Random Forest model for regression
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),  # Gradient Boosting for improved accuracy
    'KNN Regressor': KNeighborsRegressor(),  # k-Nearest Neighbors regression model
    'SVR': SVR()  # Support Vector Regression model
}

# Candidate numbers of PCA components (you can adjust these based on analysis requirements)
pca_components = [25, 30, 35, 37]  # Different PCA component configurations to test

# === Hyperparameter grids (PCA + model parameters) ===
param_grids = {
    # 1) Linear Regression – variations with and without PCA
    'Linear Regression': [
        # Case 1: No PCA applied directly
        {
            'preprocessor__num__pca': ['passthrough']  # Pass input data without transformation
        },
        # Case 2: Apply PCA with tunable number of components
        {
            'preprocessor__num__pca': [PCA()],  # Use PCA for dimensionality reduction
            'preprocessor__num__pca__n_components': pca_components  # Test with different PCA component sizes
        }
    ],

    # 2) Ridge Regression – tuning with and without PCA alongside alpha adjustment
    'Ridge': [
        # Case 1: No PCA applied
        {
            'preprocessor__num__pca': ['passthrough'],  # No transformation
            'regressor__alpha': [0.1, 1.0, 10.0]  # Regularization strength values to test
        },
        # Case 2: PCA with tunable components
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__alpha': [0.1, 1.0, 10.0]
        }
    ],

    # 3) Lasso Regression – adjusting PCA and alpha
    'Lasso': [
        # Case 1: No PCA applied
        {
            'preprocessor__num__pca': ['passthrough'],
            'regressor__alpha': [0.01, 0.1, 1.0]  # Lasso penalty strength values to test
        },
        # Case 2: PCA with tunable components
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__alpha': [0.01, 0.1, 1.0]
        }
    ],

    # 4) Random Forest – considering PCA and various hyperparameters
    'Random Forest': [
        # Case 1: No PCA
        {
            'preprocessor__num__pca': ['passthrough'],  # Pass data without PCA
            'regressor__n_estimators': [100, 200],  # Number of trees in the forest
            'regressor__max_depth': [None, 10, 20],  # Depth of trees
            'regressor__min_samples_split': [2, 5],  # Minimum samples to split a node
            'regressor__min_samples_leaf': [1, 2]  # Minimum samples at leaf node
        },
        # Case 2: PCA applied
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__n_estimators': [100, 200],
            'regressor__max_depth': [None, 10, 20],
            'regressor__min_samples_split': [2, 5],
            'regressor__min_samples_leaf': [1, 2]
        }
    ],

    # 5) Gradient Boosting – PCA on/off + GB hyperparams
    'Gradient Boosting': [
        # Case 1: no PCA
        {
            'preprocessor__num__pca': ['passthrough'],
            'regressor__n_estimators': [100, 200],
            'regressor__learning_rate': [0.05, 0.1],
            'regressor__max_depth': [3, 5]
        },
        # Case 2: PCA
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__n_estimators': [100, 200],
            'regressor__learning_rate': [0.05, 0.1],
            'regressor__max_depth': [3, 5]
        }
    ],

    # 6) KNN – PCA on/off + k, weights
    'KNN Regressor': [
        # Case 1: no PCA
        {
            'preprocessor__num__pca': ['passthrough'],
            'regressor__n_neighbors': [3, 5, 10],
            'regressor__weights': ['uniform', 'distance']
        },
        # Case 2: PCA
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__n_neighbors': [3, 5, 10],
            'regressor__weights': ['uniform', 'distance']
        }
    ],

    # 7) SVR – PCA on/off + C, gamma, kernel
    'SVR': [
        # Case 1: no PCA
        {
            'preprocessor__num__pca': ['passthrough'],
            'regressor__C': [1, 10],
            'regressor__gamma': ['scale', 0.01],
            'regressor__kernel': ['rbf']
        },
        # Case 2: PCA
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__C': [1, 10],
            'regressor__gamma': ['scale', 0.01],
            'regressor__kernel': ['rbf']
        }
    ]
}



# 3. Model Evaluation Function with Optional Hyperparameter Tuning and PCA Detection

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, preprocessor, model_name=None):
    """
    Build a pipeline, fit it, evaluate metrics, and optionally perform GridSearchCV.
    Uses num_cores (all but one CPU) for parallel processing.
    """

    # Create a pipeline that combines preprocessing and the regression model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),  # Add the preprocessing step to the pipeline
        ('regressor', model)  # Add the regression model to the pipeline
    ])

    used_pca = False  # Flag to indicate if PCA was used; default is False

    # -------------------------
    # 1) Models WITH tuning
    # -------------------------
    # Check if tuning is required for the specified model
    if model_name and model_name in param_grids:
        # Setup GridSearchCV to optimize hyperparameters
        grid = GridSearchCV(
            estimator=pipeline,  # Use the defined pipeline as the estimator
            param_grid=param_grids[model_name],  # Parameter grid for the specific model
            scoring='neg_mean_absolute_error',  # Score metric to minimize (MAE)
            cv=5,  # Use 5-fold cross-validation
            n_jobs=num_cores  # Use multiple CPU cores for parallel processing
        )

        start_time = time.time()  # Record the start time for training
        grid.fit(X_train, y_train)  # Fit the model on the training data
        training_time = time.time() - start_time  # Calculate the total training time

        best_pipeline = grid.best_estimator_  # Extract the best pipeline from GridSearch
        model_used = best_pipeline.named_steps['regressor']  # Get the best regression model

        # Calculate cross-validated MAE and standard deviation from GridSearchCV results
        cv_mae = -grid.best_score_  # Convert negative MAE to positive
        cv_std = grid.cv_results_['std_test_score'][grid.best_index_]  # Get std of best score

        # >>> NEW: Detect whether PCA was actually used <<<
        try:
            # Access the preprocessor component of the best pipeline
            preproc = best_pipeline.named_steps['preprocessor']
            num_pipe = preproc.named_transformers_['num']  # Get the numeric transformer
            pca_step = num_pipe.named_steps.get('pca', 'passthrough')  # Check if PCA is in the pipeline
            used_pca = pca_step != 'passthrough'  # Update used_pca flag if PCA is used
        except Exception:
            used_pca = False  # If PCA detection fails, keep the default as False

        # (optional) Print best parameters found during tuning for inspection
        # print(f"Best params for {model_name}: {grid.best_params_}")

    # -------------------------
    # 2) Models WITHOUT tuning
    # -------------------------
    else:
        start_time = time.time()  # Record start time for training
        pipeline.fit(X_train, y_train)  # Fit the pipeline on training data without tuning
        training_time = time.time() - start_time  # Calculate training time

        best_pipeline = pipeline  # For non-tuned models, this is the best pipeline
        model_used = model  # The model used remains the one provided as input

        # Perform cross-validation to obtain MAE scores
        cv_scores = cross_val_score(
            best_pipeline,
            X_train,
            y_train,
            cv=5,  # Use 5-fold cross-validation
            scoring='neg_mean_absolute_error',  # Scoring metric
            n_jobs=num_cores  # Use multiple CPU cores for parallel processing
        )
        cv_mae_scores = -cv_scores  # Convert negative MAE scores to positive values
        cv_mae = cv_mae_scores.mean()  # Calculate the mean of the MAE scores
        cv_std = cv_mae_scores.std()  # Calculate the standard deviation of the MAE scores

        # For non-tuned models, the PCA usage should match what was defined in the preprocessor;
        # The default value of used_pca as False is acceptable since tuning was not performed.
    # -------------------------
    # 3) Train/test metrics
    # -------------------------
    # Predict target values for both training and test sets using the best pipeline
    y_pred_train = best_pipeline.predict(X_train)  # Predictions on the training data
    y_pred_test = best_pipeline.predict(X_test)  # Predictions on the test data

    # Calculate evaluation metrics for the training set
    train_mae = mean_absolute_error(y_train, y_pred_train)  # Mean Absolute Error on training data

    # Calculate evaluation metrics for the test set
    test_mae = mean_absolute_error(y_test, y_pred_test)  # Mean Absolute Error on test data

    # Compute Root Mean Squared Error for training data
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))  # RMSE on training data

    # Compute Root Mean Squared Error for test data
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))  # RMSE on test data

    # Calculate R² score for the training set (indicates how well the model explains the variance)
    train_r2 = r2_score(y_train, y_pred_train)  # R² score for training predictions

    # Calculate R² score for the test set
    test_r2 = r2_score(y_test, y_pred_test)  # R² score for test predictions

    # Return a dictionary containing the evaluation results and relevant information
    return {
        'model': model_used,  # The model used for predictions
        'train_mae': train_mae,  # Training Mean Absolute Error
        'test_mae': test_mae,  # Test Mean Absolute Error
        'train_rmse': train_rmse,  # Training Root Mean Squared Error
        'test_rmse': test_rmse,  # Test Root Mean Squared Error
        'train_r2': train_r2,  # Training R² score
        'test_r2': test_r2,  # Test R² score
        'cv_mae': cv_mae,  # Cross-validated Mean Absolute Error
        'cv_std': cv_std,  # Standard deviation of cross-validated MAE
        'training_time': training_time,  # Total time taken for training
        'pipeline': best_pipeline,  # Pipeline used for fitting and predictions
        'y_pred_test': y_pred_test,  # Predictions made on test data
        'used_pca': used_pca  # Flag indicating if PCA was used in the preprocessing
    }


# 4. Initialization of Result Storage and Target Configuration for Model Evaluation

In [ ]:
# Use datasets dictionary directly (already contains all 6 datasets)
all_results = {}  # Initialize an empty dictionary to store results from model evaluations

# Configuration for the forecasting horizon and target variable
horizon = "24h"  # Define the time horizon for predictions (e.g., 24 hours)
target_col = "target_tre200h0_plus24h"  # Specify the target column to predict during model evaluation


# 5. Modification of Dataset Evaluation Loop to Include Multiple Datasets

Define which datasets to evaluate
datasets_to_run = ['drop_na', 'mean_imputed', 'median_imputed']  # List of datasets for evaluation

Iterate through each dataset in the datasets dictionary
for name, dataset in datasets.items():
    print(f"\n=== Evaluating models on dataset: {name} ===")

    # Check if the current dataset is in the list of datasets to run
    if name not in datasets_to_run:  # If the dataset is not in the specified list
        print(f"Skipping {name} (not in datasets_to_run).")  # Print a message and skip this dataset
        continue  # Continue to the next dataset

    # (Continue with the evaluation process for datasets included in datasets_to_run)


# Run all models on selectedall imputed datasets, store ALL models + the BEST model

In [ ]:
# Make sure evaluate_model, models, datasets, horizon, target_col, param_grids, num_cores are already defined

# 1) Define which datasets to run over
#    Here we just take ALL keys from the datasets dict, e.g. ['drop_na', 'mean_imputed', 'median_imputed']
datasets_to_run = list(datasets.keys())

print(f"\n⚙️ Running modelling for dataset(s): {datasets_to_run}")

# Container for metrics (per dataset, per model)
# Example access later: all_results['drop_na']['Random Forest']['test_mae']
all_results = {}

# Container for ALL fitted pipelines (for reuse / inspection)
# Example access: all_pipelines[('drop_na', 'Random Forest')]
all_pipelines = {}

# Container for the single BEST model globally (by lowest Test MAE)
best_model_info = None   # will store: {dataset, model_name, metrics, pipeline}
best_mae = np.inf        # start with +∞ so that any real MAE is better

# 2) Loop over each dataset variant (drop_na, mean_imputed, median_imputed, etc.)
for name in datasets_to_run:
    # Safety check: the dataset must exist in the datasets dict
    if name not in datasets:
        print(f"❌ Dataset '{name}' not found in datasets. Skipping.")
        continue

    print(f"\n=== Evaluating models on dataset: {name} ===")

    # --- 3. Copy dataset and basic info ---
    data_model = datasets[name].copy()  # work on a copy so we don't mutate the original
    print("Modelling dataset shape:", data_model.shape)
    print("Total remaining NAs:", data_model.isna().sum().sum())

    # --- 4. Define predictors and target ---
    # Exclude all possible target columns from the features
    all_predictors = [
        col for col in data_model.columns
        if col not in [
            "target_tre200h0_plus12h",
            "target_tre200h0_plus24h",
            "target_tre200h0_plus48h"
        ]
    ]

    # Feature matrix and target vector
    X = data_model[all_predictors].copy()
    y = data_model[target_col].copy()   # e.g. "target_tre200h0_plus24h"

    # Remove duplicated columns in X (can happen after merges; sklearn doesn't like duplicates)
    X = X.loc[:, ~X.columns.duplicated()]

    # --- 5. Clean target: drop rows with NaN in y ---
    mask = ~y.isna()      # boolean mask: True where target is not NaN
    X, y = X.loc[mask], y.loc[mask]

    print("Shape of X:", X.shape)
    print("Shape of y:", y.shape)
    print(f"Selected horizon (target_col): {target_col} | Horizon label: {horizon}")

    # --- 6. Identify feature types for ColumnTransformer ---
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

    print("Numerical features:", len(numeric_features))
    print("Categorical features:", categorical_features)

    # --- 7. Preprocessor compatible with param_grids (includes a 'pca' step) ---

    # Decide the numeric imputer based on which dataset we are using:
    # - median_imputed → use median
    # - others (drop_na, mean_imputed, etc.) → use mean
    if name == 'median_imputed':
        num_imputer = SimpleImputer(strategy='median')
    else:
        num_imputer = SimpleImputer(strategy='mean')

    # Numeric pipeline:
    #  - impute
    #  - placeholder PCA step (GridSearchCV can swap 'passthrough' with PCA())
    numeric_pipeline = Pipeline(steps=[
        ('imputer', num_imputer),
        ('pca', 'passthrough')   # tuned in param_grids if you want to use PCA
    ])

    # Categorical pipeline:
    #  - impute most frequent category
    #  - one-hot encode; drop first category to avoid perfect collinearity
    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
    ])

    # ColumnTransformer: apply numeric pipeline to numeric_features
    # and categorical pipeline to categorical_features
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_pipeline, numeric_features),
            ('cat', categorical_pipeline, categorical_features)
        ],
        remainder='drop'   # any columns not listed above will be dropped
    )

    # --- 8. Train–test split (hold-out set for final evaluation) ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,      # 80% train, 20% test
        random_state=42     # fixed seed for reproducibility
    )
    print("Training set shape:", X_train.shape)
    print("Test set shape:", X_test.shape)

    # --- 9. Model evaluation loop over ALL candidate models ---
    results = {}  # will store results for this dataset: {model_name: result_dict}

    for model_name, model in models.items():
        print(f"\nEvaluating {model_name} on '{name}' (imputation type: {name})...")

        # evaluate_model:
        #  - builds full Pipeline(preprocessor + model)
        #  - runs GridSearchCV if param_grids[model_name] is defined
        #  - returns a dict with metrics and the fitted pipeline
        result = evaluate_model(
            model,
            X_train, X_test, y_train, y_test,
            preprocessor,
            model_name=model_name
        )

        # Store metrics for this model on this dataset
        results[model_name] = result

        # Store the fitted pipeline separately for later reuse
        all_pipelines[(name, model_name)] = result["pipeline"]

        # Update global best model (across ALL datasets and ALL models) by lowest Test MAE
        if result["test_mae"] < best_mae:
            best_mae = result["test_mae"]
            best_model_info = {
                "dataset": name,
                "model_name": model_name,
                "metrics": result,
                "pipeline": result["pipeline"]
            }

        # Verbose metrics: easy to compare models/datasets
        print(f"Used PCA:       {result.get('used_pca', False)}")
        print(f"MAE (train):    {result['train_mae']:.3f}")
        print(f"MAE (test):     {result['test_mae']:.3f}")
        print(f"RMSE (test):    {result['test_rmse']:.3f}")
        print(f"R² (test):      {result['test_r2']:.3f}")
        print(f"CV MAE:         {result['cv_mae']:.3f} (±{result['cv_std']:.3f})")
        print(f"Training time:  {result['training_time']:.2f} s")

    # --- 10. Save all results for this dataset in all_results ---
    all_results[name] = results

# ===== After the loop: high-level summary =====
print("\n✅ Finished evaluating dataset(s).")
print("Saved datasets in all_results:", list(all_results.keys()))

print("\n📦 Stored pipelines for (dataset, model) pairs:", len(all_pipelines))

print("\n⭐ Best model overall (by MAE):")
print("  Dataset:   ", best_model_info["dataset"])
print("  Model:     ", best_model_info["model_name"])
print("  Best MAE:  ", best_mae)


# Build the summary DataFrame (all datasets, all models)

In [ ]:
summary_rows = []  # will collect one dict (row) per (dataset, model)

for dataset_name, model_results in all_results.items():
    # model_results is a dict: {model_name: result_dict}
    for model_name, result in model_results.items():
        summary_rows.append({
            # --- Meta info about the run ---
            "Horizon": horizon,           # e.g. "24h"
            "Target_Col": target_col,     # e.g. "target_tre200h0_plus24h"
            "Dataset": dataset_name,      # e.g. "drop_na" or "median_imputed"
            "Used_PCA": result.get("used_pca", False),  # whether PCA was actually used in the best pipeline

            # --- Model identifier ---
            "Model": model_name,          # e.g. "Random Forest", "Ridge", etc.

            # --- Metrics (train / test / CV) ---
            "MAE_Train": result.get("train_mae", np.nan),   # training MAE
            "MAE_Test": result.get("test_mae", np.nan),     # test MAE (main selection metric)
            "RMSE_Test": result.get("test_rmse", np.nan),   # test RMSE
            "R²_Test": result.get("test_r2", np.nan),       # test R²

            "CV_MAE_Mean": result.get("cv_mae", np.nan),    # cross-validated MAE (mean over folds)
            "CV_MAE_Std": result.get("cv_std", np.nan),     # std dev of CV MAE

            # --- Timing ---
            "Training_Time_s": result.get("training_time", np.nan),  # wall-clock training time in seconds
        })

# If nothing was added, all_results is probably empty
if not summary_rows:
    print("⚠️ No results found in all_results. Did you run the evaluation loop?")
else:
    # Convert list of dicts to a DataFrame
    results_df = pd.DataFrame(summary_rows)

    # Sort rows: best first by lowest Test MAE, then by lowest Test RMSE
    results_df = results_df.sort_values(
        by=["MAE_Test", "RMSE_Test"]
    ).reset_index(drop=True)

    # Round numeric columns for nicer, compact display
    results_df = results_df.round(3)

    # Display in notebook (fallback to print if display is not defined)
    try:
        display(results_df)
    except NameError:
        print(results_df)


## Save the objects to disk

In [ ]:
import joblib   # good for saving sklearn models and pipelines
import pickle   # general-purpose Python object serialisation

# 1) Save all_results (metrics etc.) as a pickle file
#    - Contains metrics, CV scores, training times, and the pipeline in each result dict.
with open("all_results.pkl", "wb") as f:   # open file in write-binary mode
    pickle.dump(all_results, f)            # serialise all_results into this file

# 2) Save all_pipelines (ALL fitted pipelines) with joblib
#    - all_pipelines[(dataset_name, model_name)] -> fitted Pipeline
joblib.dump(all_pipelines, "all_pipelines.joblib")  # joblib is efficient for sklearn objects

# 3) Save best_model_info (only the globally best model)
#    - Contains: dataset name, model name, metrics dict, and the best pipeline
with open("best_model_info.pkl", "wb") as f:  # open file in write-binary mode
    pickle.dump(best_model_info, f)           # serialise best_model_info into this file

print("Saved: all_results.pkl, all_pipelines.joblib, best_model_info.pkl")


# Inspect predictions of the best model (selected by lowest test MAE)

In [ ]:
# --- 0) Pick the best model (lowest Test MAE) from all_results ---

best_dataset_name = None   # name of the dataset giving the best model (e.g. "drop_na")
best_model_name = None     # name of the model (e.g. "Random Forest")
best_mae = np.inf          # start with +∞ so any real MAE is better
best_entry = None          # will hold the full result dict for the best model

for ds_name, ds_results in all_results.items():
    # ds_results: dict {model_name: result_dict} for a given dataset
    for model_name, res in ds_results.items():
        # We assume each `res` dict has keys: "test_mae" and "pipeline"
        if res["test_mae"] < best_mae:
            best_mae = res["test_mae"]
            best_dataset_name = ds_name
            best_model_name = model_name
            best_entry = res            # store the whole result dict for later

print(f"Best dataset (by MAE): {best_dataset_name}")
print(f"Best model (by MAE): {best_model_name}")
print(f"Best MAE (test): {best_mae:.3f}")

# --- 1) Recreate X, y and the test set for that dataset ---

# List of all possible target columns, so we can exclude them from predictors
targets = [
    "target_tre200h0_plus12h",
    "target_tre200h0_plus24h",
    "target_tre200h0_plus48h"
]

# Recover the dataset that produced the best model
best_data = datasets[best_dataset_name].copy()

# Same predictors logic as in the training loop: drop all target columns
all_predictors = [
    col for col in best_data.columns
    if col not in targets
]

# Features and target
X = best_data[all_predictors].copy()
y = best_data[target_col].copy()   # same horizon as used in training (e.g. target_tre200h0_plus24h)

# Drop rows with NaN in the target (same cleaning logic as training loop)
mask = ~y.isna()            # keep only rows where y is not NaN
X = X.loc[mask].copy()
y = y.loc[mask].copy()

# Remove duplicated columns in X, just like you did before training
X = X.loc[:, ~X.columns.duplicated()]

# Re-create the SAME train–test split as during training (same random_state!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

# --- 2) Use the best pipeline to predict on this internal test set ---

best_pipeline = best_entry["pipeline"]  # fitted Pipeline(preprocessor + model) from evaluate_model

# Generate predictions on the hold-out test set
y_pred = best_pipeline.predict(X_test)

# --- 3) Build an inspection DataFrame for detailed error analysis ---

inspect_df = pd.DataFrame({
    "y_true": y_test.values,   # ground truth
    "y_pred": y_pred           # model predictions
})

# Error = prediction - truth
inspect_df["error"] = inspect_df["y_pred"] - inspect_df["y_true"]
inspect_df["abs_error"] = inspect_df["error"].abs()  # absolute error for ranking

# First few predictions vs. truth (quick sanity check)
display(inspect_df.head(10))

# Worst predictions (largest absolute error) – where the model struggles most
display(inspect_df.sort_values("abs_error", ascending=False).head(5))


In [ ]:
# Importing the test dataset
test = pd.read_csv("ML_WP_data/test.csv")
# Checking the table: weather data from Bern
test.head()

# Calculate total number of NaN values in the DataFrame
total_test_nans = test.isna().sum().sum()

# Display the total count of NaN values
print("Total NaN values in the DataFrame:", total_test_nans)


# Build Kaggle submission using best model (by Test MAE)

In [ ]:
# === 1) Find best model globally by lowest Test MAE ===

if not all_results:
    raise ValueError("all_results is empty. Run the evaluation loop first.")

best_entry = None  # will store info about the single best (dataset, model) combo

for dataset_name, model_results in all_results.items():
    # model_results is a dict: {model_name: result_dict}
    for model_name, result in model_results.items():
        # Safety: only consider entries that actually have both test_mae and pipeline
        if "test_mae" not in result or "pipeline" not in result:
            continue

        # Update if this is the first valid entry or if it has a lower Test MAE
        if best_entry is None or result["test_mae"] < best_entry["test_mae"]:
            best_entry = {
                "Dataset": dataset_name,          # which dataset (e.g. "drop_na")
                "Model": model_name,              # which model (e.g. "Random Forest")
                "test_mae": result["test_mae"],   # best Test MAE so far
                "pipeline": result["pipeline"]    # fitted Pipeline(preprocessor + model)
            }

if best_entry is None:
    # If still None, nothing had test_mae/pipeline → modelling step not run correctly
    raise ValueError("No valid model with test_mae/pipeline found in all_results.")

# Extract the fitted pipeline of the best model
best_pipeline = best_entry["pipeline"]
print(f"Best overall model: {best_entry['Model']} trained on '{best_entry['Dataset']}'")
print(f"Test MAE: {best_entry['test_mae']:.3f}")

# === 2) Define targets and predictors based on the best training dataset ===

targets = [
    "target_tre200h0_plus12h",
    "target_tre200h0_plus24h",
    "target_tre200h0_plus48h"
]

best_dataset_name = best_entry["Dataset"]          # e.g. "drop_na"
best_dataset = datasets[best_dataset_name].copy()  # training dataset used to fit this best model

# Rebuild the list of predictor columns exactly as in the training loop:
# take all columns except the target(s)
all_predictors = [
    col for col in best_dataset.columns
    if col not in targets
]

# === 3) Add missing_count to Kaggle test if the training data had it ===
# (optional engineered feature: only add it to test if it was present in training)

if "missing_count" in best_dataset.columns and "missing_count" not in test.columns:
    test["missing_count"] = test.isna().sum(axis=1)   # row-wise count of NaNs
    print("Added missing_count column to Kaggle test dataset.")

# === 4) Align predictors between training and Kaggle test ===

# Keep only predictors that also exist in the Kaggle test DataFrame
available_predictors = [c for c in all_predictors if c in test.columns]
missing_cols = set(all_predictors) - set(available_predictors)

if missing_cols:
    # Inform you if some training columns are not present in the Kaggle test
    print(
        "Warning: these training columns are missing in the Kaggle test set "
        "and will be ignored: "
        f"{missing_cols}"
    )

# Final feature matrix for Kaggle predictions
X_kaggle = test[available_predictors].copy()

# Note:
# No external scaling/imputation here.
# best_pipeline already contains the full preprocessor
# (imputer, scaler, PCA, encoders, etc.) fitted on the training data.

# === 5) Predict on Kaggle test set using the best pipeline ===

y_pred_24 = best_pipeline.predict(X_kaggle)   # predictions for target_tre200h0_plus24h

# === 6) Build and save Kaggle submission file ===

# Build submission DataFrame with the correct column names
submission = pd.DataFrame({
    "Id": test["Id"],                         # Id column from Kaggle test
    "target_tre200h0_plus24h": y_pred_24      # predicted target for 24h horizon
})

# Validate submission format – only keep expected columns
expected_cols = ["Id", "target_tre200h0_plus24h"]
extra_cols = set(submission.columns) - set(expected_cols)
if extra_cols:
    print(f"Warning: removing unexpected columns from submission: {extra_cols}")
    submission = submission[expected_cols]

# Save file with a date-stamped filename
day = date.today().strftime("%Y%m%d")
filename = f"weather_report_submission_{day}.csv"
submission.to_csv(filename, index=False, encoding="utf-8")

print(f"\nSaved Kaggle submission as: {filename}")
print(
    f"Best model: {best_entry['Model']} | "
    f"Dataset: {best_entry['Dataset']} | "
    f"Test MAE: {best_entry['test_mae']:.3f}"
)

# Quick look at the first few rows of the submission
display(submission.head())


In [ ]:
print("Training target stats (y):")
print(datasets["drop_na"][target_col].describe())

print("\nKaggle predictions stats:")
print(pd.Series(y_pred_24).describe())
